<a href="https://colab.research.google.com/github/Shxnxz/movie-recommendation-model/blob/main/Ensemble_Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install library

In [ ]:
#Might have to restart session when importing older version of numpy
import sys

!pip install "numpy<2.0"
!pip install --no-cache-dir scikit-surprise

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 14.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp312-cp312-linux_x86_64.whl size=2554968 sha256=e81352a479fbeec0573210fe651b1d86451b211b679d8402babe25561f40fcc1
  Stored in directory: /tmp/pip-ephem-wheel-cache-mv3tzh63/wheels/75/fa/bc/739bc2cb1fbaab6061854e6cfbb81a0ae52c92a502a7fa454b
Successfully built scikit-surprise


Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load the Datasets

train_df = pd.read_csv('train.csv')

movies_cols = ['MovieID', 'Title', 'Genres']
movies_df = pd.read_csv('movies.dat', sep='::', engine='python', names=movies_cols, encoding='latin-1')

val_df = pd.read_csv("val.csv")

#Preview Data
#train_df.head()
#movies_df.head()
#val_df.head()



Why we Use The Latent Factor Model (SVD)

In [ ]:
# Set visual style
sns.set_theme(style="whitegrid")

print("Matrix Sparisty Calculation")
# Calculate how empty the user-item interaction matrix is
n_users = train_df['user_id'].nunique()
n_movies = train_df['movie_id'].nunique()
n_ratings = len(train_df)

total_possible_interactions = n_users * n_movies
sparsity = (1.0 - (n_ratings / total_possible_interactions)) * 100

print(f"Total Users: {n_users}")
print(f"Total Movies Rated: {n_movies}")
print(f"Total Ratings: {n_ratings}")
print(f"Matrix Sparsity: {sparsity:.2f}%")
print(f"{sparsity:.2f}% sparse matrix proves why User-Based system approaches struggle, justifying our use of Matrix Factorization (SVD) to find latent features.\n")

Why we use Collaborative Filtering (Item-Based System)

In [ ]:
print("Long Tail Distribution")
# Plot how many ratings each movie gets
ratings_per_movie = train_df.groupby('movie_id').size().sort_values(ascending=False)

plt.figure(figsize=(9, 5))
plt.plot(ratings_per_movie.values, color='red')
plt.title('Long-Tail Distribution of Movie Ratings')
plt.xlabel('Sorted Movie Rank (0 = Most Popular)')
plt.ylabel('Number of Ratings')
plt.fill_between(range(len(ratings_per_movie)), ratings_per_movie.values, color='red', alpha=0.3)
plt.show()

print("A massive skew towards a few movies justifies testing a Popularity baseline.")
print("The long, flat tail justifies Item-Based KNN over User-Based KNN")
print("as item similarities are more stable across sparse data.\n")

Why we use Content-Based Recommendation (TF-IDF)

In [ ]:
print("GENRE FREQUENCY")
# Explode the genres separated by '|' and count them
all_genres = movies_df['Genres'].str.split('|', expand=True).stack().value_counts()

plt.figure(figsize=(10, 5))
sns.barplot(x=all_genres.values, y=all_genres.index, hue=all_genres.index, palette="viridis", legend=False)
plt.title('Distribution of Movie Genres in Dataset')
plt.xlabel('Number of Movies')
plt.ylabel('Genre')
plt.show()

print("Clear categorical biases (e.g., heavily skewed toward Drama/Comedy) provide a strong, independent signal.")
print("This justifies building a Content-Based TF-IDF model to handle 'cold-start' niche movies that lack rating history.")

SVD Implementation

In [ ]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split as surprise_split
from surprise.model_selection import GridSearchCV

# Reader defines the rating scale so Surprise can clip predictions correctly. (1-5)
reader = Reader(rating_scale=(1, 5))

# load_from_df expects exactly three columns: user, item, rating.
surprise_data = Dataset.load_from_df(
    train_df[["user_id", "movie_id", "rating"]],
    reader
)
### Use Grid Search to find Optimal Parameter

# param_grid = {
#     "n_factors": [50, 100, 150],
#     "n_epochs":  [20, 30],
#     "lr_all":    [0.005, 0.01],
#     "reg_all":   [0.02, 0.05],
# }

### Running this chunk took 30 mins we only need to run it once after we obtained the Value
# gs = GridSearchCV(
#     SVD,
#     param_grid,
#     measures=["rmse"],
#     cv=5,
#     n_jobs=-1
#     )

# gs.fit(surprise_data)

# print("Best RMSE Score:", gs.best_score['rmse'])     # best CV RMSE
# print("Optimal Parameters:", gs.best_params['rmse']) # best parameters

### Result
# Best RMSE Score: 0.8540660912780847
# Optimal Parameters: {'n_factors': 150, 'n_epochs': 20, 'lr_all': 0.01, 'reg_all': 0.05}
svd_model = SVD(
    n_factors=150,
    n_epochs=20,
    lr_all=0.01,
    reg_all=0.05,
    random_state=42
)

# Build a full Trainset from ALL training data (no internal split).
trainset_full = surprise_data.build_full_trainset()
svd_model.fit(trainset_full)

def get_svd_prediction(user_id: int, movie_id: int) -> float:
  prediction = svd_model.predict(uid=user_id, iid=movie_id)
  #svd_model.predict() returns a Prediction object
  #eg. user: 101  item: 55  r_ui: None  est: 4.12  details: {'was_impossible': False}
  #Adding .est at the end extracts only the actual floating-point prediction
  return prediction.est

SVD Test using sample from train.df

In [ ]:
_sample = train_df.iloc[6767]
print(f" SVD prediction for "
      f"user={_sample.user_id}, movie={_sample.movie_id}: "
      f"{get_svd_prediction(_sample.user_id, _sample.movie_id):.3f} "
      f"(actual: {_sample.rating})")

Item-Based Knn Implementation

In [ ]:
from surprise import KNNWithMeans

### Use Grid Search to find Optimal Parameter

# param_grid = {
# "k":   [20, 40, 60, 80],     # Max number of neighbors
# "min_k": [1, 3, 5],
# "sim_options": {
#         "name": ["pearson_baseline", "cosine", "pearson"],
#         "user_based": [False]
#     }
# }

### Same as SVD implmentation we only need to run it once after we obtained the Value
# gs_knn = GridSearchCV(
#     KNNWithMeans,
#     param_grid,
#     measures=["rmse"],
#     cv=5,
#     n_jobs=-1
#     )

# print("\n--- KNN Optimization Complete ---")
# print(f"Best RMSE: {gs_knn.best_score['rmse']}") # best CV RMSE
# print(f"Best Parameters: {gs_knn.best_params['rmse']}") # best parameters

# Best RMSE: 0.8569623268208341
# Best Parameters: {'k': 20, 'min_k': 5, 'sim_options': {'name': 'pearson_baseline', 'user_based': False}}

# sim_options = {
#     "name": "pearson_baseline",   # bias-corrected Pearson correlation
#     "user_based": False,          # ← ITEM-BASED (critical for this task)
#     "shrinkage": 100,             # shrink similarities when co-rated count is low
#     "min_support": 5,             # min co-ratings needed to compute similarity
# }

knn_sim_options = {
    'name': 'pearson_baseline',
    'user_based': False
}

knn_model = KNNWithMeans(
    k=20,                         # max neighbors to consider
    min_k=5,                      # fall back to item mean if < min_k neighbors
    sim_options=knn_sim_options,
    verbose=False
)
knn_model.fit(trainset_full)      # reuse the same full trainset

def get_knn_prediction(user_id: int, movie_id: int) -> float:
  prediction = knn_model.predict(uid=user_id, iid=movie_id)
  return prediction.est

KNN Test using sample from train.df

In [ ]:
_sample = train_df.iloc[6767]
print(f" KNN prediction for "
      f"user={_sample.user_id}, movie={_sample.movie_id}: "
      f"{get_knn_prediction(_sample.user_id, _sample.movie_id):.3f} "
      f"(actual: {_sample.rating})")

Content-Based Implementation

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Replace '|' with ' ' so each genre becomes a separate token
# e.g. "Action|Comedy|Romance" → "Action Comedy Romance"
movies_df["genre_str"] = movies_df["Genres"].fillna("").str.replace("|", " ", regex=False)

# Fit TF-IDF on all movie genre strings
# - No stop_words: genre labels are not prose, nothing should be removed
# - min_df=1: keep all genres including rare ones like "Film-Noir", "Western"
tfidf = TfidfVectorizer(min_df=1)
tfidf_matrix = tfidf.fit_transform(movies_df["genre_str"])

# Convert to dense numpy array for fast slicing
# Shape: (n_movies, n_genre_features)
tfidf_dense = tfidf_matrix.toarray()

# Build O(1) lookup: MovieID → row index in tfidf_dense
movieid_to_idx = {
    int(row.MovieID): idx
    for idx, row in movies_df.iterrows()
}

print(f"TF-IDF matrix: {tfidf_dense.shape[0]} movies × {tfidf_dense.shape[1]} genre features")

#BUILD USER TASTE PROFILES

# Precompute global fallback — done once here, never inside the function
OVERALL_AVG = float(train_df["rating"].mean())

# Filter to positive ratings only
positive_ratings = train_df[train_df["rating"] > 3].copy()

def _build_user_profile(user_ratings: pd.DataFrame) -> np.ndarray | None:

    weighted_sum  = np.zeros(tfidf_dense.shape[1])
    total_weight  = 0.0

    for _, row in user_ratings.iterrows():
        mid = int(row["movie_id"])           # camelCase — matches train.csv
        if mid not in movieid_to_idx:
            continue                         # movie missing from metadata, skip

        weight        = float(row["rating"])
        weighted_sum += weight * tfidf_dense[movieid_to_idx[mid]]
        total_weight += weight

    if total_weight == 0:
        return None                          # no usable ratings for this user

    return weighted_sum / total_weight       # normalised profile vector

user_profiles = {}
for uid, group in positive_ratings.groupby("user_id"):
    profile = _build_user_profile(group)
    if profile is not None:
        user_profiles[int(uid)] = profile

print(f"Profiles built for {len(user_profiles):,} users.")
print(f"Global average rating (fallback): {OVERALL_AVG:.3f}")


# PREDICTION FUNCTION


def get_cbf_prediction(user_id: int, movie_id: int) -> float:
    # ── Cold-start: unknown user ───────────────────────────────────────────
    user_id  = int(user_id)
    movie_id = int(movie_id)

    if user_id not in user_profiles or movie_id not in movieid_to_idx:
        return OVERALL_AVG

    # ── Retrieve vectors ───────────────────────────────────────────────────
    user_vec  = user_profiles[user_id]                        # (n_features,)
    movie_vec = tfidf_dense[movieid_to_idx[movie_id]]         # (n_features,)

    # ── Cosine similarity ──────────────────────────────────────────────────
    # cosine_similarity expects 2D input → reshape, then extract scalar
    dot_product = np.dot(user_vec, movie_vec)
    norm        = np.linalg.norm(user_vec) * np.linalg.norm(movie_vec)
    sim         = float(np.clip(dot_product / (norm + 1e-9), 0.0, 1.0))

    # ── Rescale to [1.0, 5.0] ─────────────────────────────────────────────
    predicted = 1.0 + float(sim) * 4.0

    return float(np.clip(predicted, 1.0, 5.0))

TF-IDF matrix: 3883 movies × 20 genre features
Profiles built for 6,035 users.
Global average rating (fallback): 3.572


Test Content-Based Rating Prediction

In [ ]:
_sample = train_df.iloc[44]
print(f" Content-Based prediction for "
      f"user={_sample.user_id}, movie={_sample.movie_id}: "
      f"{get_cbf_prediction(_sample.user_id, _sample.movie_id):.3f} "
      f"(actual: {_sample.rating})")

# DES431 PROJECT — ENSEMBLE WEIGHT SEARCH & Precision@10


### Precision@10 Calculation

Ensemble Model

In [ ]:
from itertools import product

#Grab the movie that user actually like (Rating better than 3)
#We use first here not because its the best to recommend but because val.csv contains exactly ONE relevant movie per user.
val_relevant = (
    val_df[val_df["rating"] > 3]
    .groupby("user_id")["movie_id"]
    .first()             # one per user
    .to_dict()           # {user_id: movie_id}
)

# For each user we recommend only UNSEEN movies — movies they did NOT rate
# All movie IDs that exist in the training set
all_movie_ids = set(train_df["movie_id"].unique())

# Movies each user has already seen (rated in training)
user_seen = (
    train_df.groupby("user_id")["movie_id"]
    .apply(set)
    .to_dict()   # {user_id: {movie_id, ...}}
)

# For each validation user, their unseen candidate pool = all movies - seen
val_users = list(val_relevant.keys())
user_candidates = {
    uid: list(all_movie_ids - user_seen.get(uid, set()))
    for uid in val_users
}

### pre-calculates and caches every model's scores for every possible movie recommendation

#precomputed = {}
# for i, uid in enumerate(val_users):
#     candidates = user_candidates[uid]

#     svd_scores = np.array([get_svd_prediction(uid, mid) for mid in candidates])
#     knn_scores = np.array([get_knn_prediction(uid, mid) for mid in candidates])
#     cbf_scores = np.array([get_cbf_prediction(uid, mid) for mid in candidates])

#     precomputed[uid] = {
#         "movies"    : candidates,
#         "svd_scores": svd_scores,
#         "knn_scores": knn_scores,
#         "cbf_scores": cbf_scores,
#         "relevant"  : val_relevant.get(uid)
#     }

#Precision@10
def compute_precision_at_10(w_svd: float, w_knn: float, w_cbf: float) -> float:
    hits = 0   # number of users where the relevant movie is in their top 10

    for uid, data in precomputed.items():
        relevant_movie = data["relevant"]
        if relevant_movie is None:
            continue   # no ground truth for this user, skip

        # Weighted ensemble score for each candidate movie
        ensemble_scores = (
            w_svd * data["svd_scores"] +
            w_knn * data["knn_scores"] +
            w_cbf * data["cbf_scores"]
        )

        # Get indices of top-10 scores (descending order)
        top10_indices = np.argpartition(ensemble_scores, -10)[-10:]

        # Map indices back to movie IDs
        top10_movies = [data["movies"][idx] for idx in top10_indices]

        # Check if the relevant movie appears in the top 10
        if relevant_movie in top10_movies:
            hits += 1

    n_users = len([u for u in precomputed if precomputed[u]["relevant"] is not None])
    return hits * 0.1 / n_users if n_users > 0 else 0.0

#GRID SEARCH OVER WEIGHTS

# STEP = 0.1
# best_p10     = -1.0
# best_weights = (None, None, None)
# results      = []   # store all results for analysis

# Generate all (w1, w2) pairs; w3 is determined
# weight_values = np.round(np.arange(0, 1.0 + STEP, STEP), 2)

# for w_svd, w_knn in product(weight_values, repeat=2):
#     w_cbf = round(1.0 - w_svd - w_knn, 10)   # avoid floating-point drift

#     if w_cbf < 0:
#         continue   # invalid combination

#     p10 = compute_precision_at_10(w_svd, w_knn, w_cbf)
#     results.append((w_svd, w_knn, round(w_cbf, 2), p10))

#     if p10 > best_p10:
#         best_p10    = p10
#         best_weights = (w_svd, w_knn, round(w_cbf, 2))

# results_df = pd.DataFrame(results, columns=["w_svd", "w_knn", "w_cbf", "precision_at_10"])
# results_df = results_df.sort_values("precision_at_10", ascending=False).reset_index(drop=True)

# print(f"  w_svd = {best_weights[0]}")
# print(f"  w_knn = {best_weights[1]}")
# print(f"  w_cbf = {best_weights[2]}")
# print(f"\nBest Precision@10 on val.csv: {best_p10:.4f}")

# any (user_id, movie_id) pair for generating final recommendation lists.
# w_svd = 0.6
# w_knn = 0.1
# w_cbf = 0.3
# Best Precision@10 on val.csv: 0.0421
W_SVD = 0.6
W_KNN = 0.1
W_CBF = 0.3
def get_ensemble_prediction(user_id: int, movie_id: int) -> float:
    return (
        W_SVD * get_svd_prediction(user_id, movie_id) +
        W_KNN * get_knn_prediction(user_id, movie_id) +
        W_CBF * get_cbf_prediction(user_id, movie_id)
    )

print(f"\nEnsemble function ready: get_ensemble_prediction(user_id, movie_id)")
print(f"  Weights locked to: SVD={W_SVD}, KNN={W_KNN}, CBF={W_CBF}")

KeyboardInterrupt: 

Generate Submission

In [ ]:
rows = []   # will become the output DataFrame

for i, uid in enumerate(val_users):

    # Candidate movies = all movies the user has NOT seen in training
    seen = user_seen.get(uid, set())
    candidates = list(all_movie_ids - seen)

    # Score every candidate with the ensemble
    scores = np.array([get_knn_prediction(uid, mid) for mid in candidates])

    # Get indices of top 10 scores (descending)
    # np.argpartition is faster than full sort for large candidate lists
    top10_indices = np.argpartition(scores, -10)[-10:]

    # Sort those 10 by score descending (argpartition doesn't guarantee order)
    top10_indices = top10_indices[np.argsort(scores[top10_indices])[::-1]]

    # Map back to movie IDs
    top10_movies = [candidates[idx] for idx in top10_indices]

    # Format as comma-separated string per submission spec
    recommended_str = ",".join(str(mid) for mid in top10_movies)

    rows.append({
        "user_id":             uid,
        "recommended_movies":  recommended_str
    })

    # Progress report every 500 users
    if (i + 1) % 500 == 0 or (i + 1) == len(val_users):
        print(f"  Processed {i+1}/{len(val_users)} users...")

# =============================================================================
# SECTION 3 — SAVE TO CSV
# =============================================================================

submission_df = pd.DataFrame(rows, columns=["user_id", "recommended_movies"])
submission_df.to_csv("recommendations.csv", index=False)

print(f"\nDone! File saved as recommendations.csv")
print(f"  Rows : {len(submission_df)}")
print(f"\nPreview:")
print(submission_df.head(5).to_string(index=False))


  Processed 500/6037 users...
  Processed 1000/6037 users...
  Processed 1500/6037 users...
  Processed 2000/6037 users...
  Processed 2500/6037 users...
  Processed 3000/6037 users...
  Processed 3500/6037 users...
  Processed 4000/6037 users...
  Processed 4500/6037 users...
  Processed 5000/6037 users...
  Processed 5500/6037 users...
  Processed 6000/6037 users...
  Processed 6037/6037 users...

Done! File saved as recommendations.csv
  Rows : 6037

Preview:
 user_id                               recommended_movies
       1    1830,53,3881,3607,572,3382,3656,3172,2360,439
       2   3233,53,3656,572,2480,3881,3172,3607,1830,3382
       3 3607,2480,3881,3656,572,3172,1830,3382,3233,3245
       4 1830,2930,3172,3382,3607,2019,572,2480,3881,3245
       5  3881,1830,3382,3172,3233,3656,572,2480,3607,787


Prediction@10 Using recommend list to test list

In [ ]:
import pandas as pd

# Load datasets
recommendations_df = pd.read_csv("recommendations.csv")
val_df = pd.read_csv("val.csv")

# Map each user to a set of their relevant movies from the validation set
relevant_movies_per_user = val_df.groupby('user_id')['movie_id'].apply(set).to_dict()

def calculate_precision_at_k(row, k=10):
    user_id = row['user_id']

    # If user has no ground truth data, precision is 0
    if user_id not in relevant_movies_per_user:
        return 0.0

    relevant_movies = relevant_movies_per_user[user_id]

    # Parse comma-separated string into integer list
    if isinstance(row['recommended_movies'], str):
        recommended = [int(x.strip()) for x in row['recommended_movies'].split(',')]
    else:
        return 0.0

    # Isolate top 10
    top_k_recommended = set(recommended[:k])

    # Count how many of the top 10 are actually in the user's relevant movies
    hits = len(top_k_recommended.intersection(relevant_movies))

    return hits / k

# Calculate metric for each user
recommendations_df['precision_at_10'] = recommendations_df.apply(lambda row: calculate_precision_at_k(row, 10), axis=1)

# Output summary
mean_precision = recommendations_df['precision_at_10'].mean()
print(f"Mean Precision@10: {mean_precision}")

# Save the resulting DataFrame
recommendations_df[['user_id', 'precision_at_10']].to_csv('precision_results.csv', index=False)